# CKIN2 - Souvenirs

Analyse en lecture seule des inventaires et prix normalises depuis les exports locaux.
Les prix sont des observations historiques, pas des cotations actuelles. Les dates estimees sont identifiees explicitement.
Les quantites sont celles de l'export : elles ne constituent pas des snapshots historiques.


## 1. Chargement et controle des donnees

Generer les CSV depuis la racine avec `python src/process_categories.py`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src/analyze_categories.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analyze_categories import analyze_category, split_price_records, category_price_figures

DATA_DIR = PROJECT_ROOT / "data/processed"
inventory = pd.read_csv(DATA_DIR / "souvenirs_inventory.csv")
raw_prices = pd.read_csv(DATA_DIR / "souvenirs_prices.csv")
market_records, acquisitions = split_price_records(raw_prices)
prices, valuation, unknown_quantities, coverage, summary = analyze_category(inventory, raw_prices)
print("Acquisitions : dates connues, valeurs de reference ; prix d achat non confirmes.")
display(acquisitions)
display(inventory)
display(coverage)
print("Observations avec date estimee :", market_records.date_estimated.sum())
display(market_records.loc[market_records.date_estimated.eq(True)])
if not unknown_quantities.empty:
    print("Quantites inconnues : lignes exclues de la valorisation ; total indisponible.")
    display(unknown_quantities)


## 2. Evolution des prix de marche et acquisitions

Les croix indiquent les valeurs de reference aux dates d acquisition ; elles ne sont pas reliees aux courbes de marche.
La base 100, la couverture et la valorisation utilisent uniquement les releves de marche (source_column > 0).
Les dates estimees sont indiquees au survol des points. Les acquisitions figurent aussi dans une chronologie distincte.


In [ ]:
if not prices.empty or not acquisitions.empty:
    absolute, relative = category_price_figures(prices, acquisitions)
    absolute.show()
    if not prices.empty:
        relative.show()
    else:
        print("Aucun prix de marche disponible : indices et valorisation indisponibles.")
else:
    print("Aucune observation disponible.")
if not acquisitions.empty:
    px.scatter(acquisitions, x="date", y="item_name",
               title="Dates d'acquisition - achats ou drops, sans cout confirme").show()


## 3. Valeur de l'inventaire aux derniers prix connus

Le sous-total connu exclut les quantites ou prix manquants. Le total reste indisponible si la couverture est incomplete.
Les prix d'achat sans date restent utilisables pour comparer le cout et la valeur ; aucun rendement annualise n'est calcule.


In [ ]:
display(pd.Series(summary))
display(valuation.sort_values("market_value", ascending=False))
priced = valuation.loc[valuation.market_value.notna() & valuation.quantity.gt(0)]
if not priced.empty:
    px.bar(priced.sort_values("market_value"), x="market_value", y="item_name",
           orientation="h", color="item_type", title="Contribution au sous-total connu (EUR)").show()


## 4. Prix d'achat et limites

L'ecart indique est une difference brute, sans frais de vente. Les dates d'achat sont inconnues.
Les dates de marche intermediaires sont estimees entre les dates voisines, puis ramenees au jour civil.
Une courbe de valeur historique du portefeuille exige des snapshots de quantites : appliquer les quantites actuelles au passe serait trompeur.


In [ ]:
purchases = valuation.loc[valuation.purchase_unit_price.notna(),
    ["item_name", "quantity", "purchase_unit_price", "purchase_date", "purchase_cost", "market_value", "unrealized_change"]]
if purchases.empty:
    print("Aucun prix d'achat renseigne dans cet export.")
else:
    display(purchases)
